# 01 — Kesif

**Bu defter rapor uretmez.** Projenin butun sayilari `reports/` altindaki
script ciktilarindan gelir (`python run_all.py`). Burasi, o sonuclara
bakmadan once veriye kendi gozunle bakmak icin.

Burada bulunan hicbir sey, bir script tarafindan yeniden uretilmeden
rapor iddiasi olmaz. Defter kesfeder, script kanitlar.

**Hucre ciktilari commit edilmez** (`nbstripout`, bkz.
`.pre-commit-config.yaml`). Repoda yalnizca kod ve aciklama durur;
grafikleri gormek icin defteri kendin calistirmalisin.

Gereken dosya: `data/processed/clean_v1.csv`. Yoksa once:

```
python run_all.py --only clean
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src" / "data_processing"))
sys.path.insert(0, str(ROOT / "src" / "modeling"))
import schema  # noqa: E402
import splits  # noqa: E402

PROC = ROOT / "data" / "processed" / "clean_v1.csv"
if not PROC.exists():
    raise FileNotFoundError(
        f"{PROC} yok. Once calistir: python run_all.py --only clean")

df = pd.read_csv(PROC, parse_dates=["time_stamp"])
kpi = pd.read_csv(ROOT / "reports" / "output_kpi_summary.csv")
print(f"clean_v1: {df.shape[0]:,} satir x {df.shape[1]} kolon")

## K1 — veri tek bir ~4 saatlik pencereden geliyor

14.088 satir cok gorunuyor ama bagimsiz gozlem sayisi degil. Long-term
capability, vardiya karsilastirmasi ve gun-ici trend bu veriyle
yapilamaz. Asagidaki hucre bunu dogrudan gosteriyor.

In [ ]:
span = df.time_stamp.max() - df.time_stamp.min()
step = df.time_stamp.diff().dt.total_seconds().median()

print(f"aralik      : {df.time_stamp.min()} -> {df.time_stamp.max()}")
print(f"sure        : {span}  ({span.total_seconds() / 3600:.1f} saat)")
print(f"ornekleme   : {step:.0f} sn (medyan adim)")
print(f"durus blogu : {int(df.flag_downtime.sum())} satir (R5 ile isaretli)")

## K5 — ardisik gozlemler bagimsiz degil

Karar degiskenlerinde otokorelasyon cok yuksek. Rastgele train/test
bolmesi kullanilirsa test satirinin komsulari train'de kalir, model
tahmin degil hatirlama yapar ve R2 sahte sekilde yukselir.

Embargo genisligi bu yuzden veriden hesaplaniyor: otokorelasyonun 0.2
altina indigi mesafenin medyani. **SANSURLU** satirlar, taramanin ust
sinirina kadar hic sonumlenmeyen serilerdir; donen sayi bir olcum degil,
aramanin durdugu yerdir.

In [ ]:
dvs = [c for c in schema.decision_variables(df.columns) if c in df.columns]

for col in dvs[:8]:
    lag, censored = splits.acf_decay_lag(df[col], return_censored=True)
    durum = "SANSURLU (tarama siniri)" if censored else "olculdu"
    print(f"{col.replace('.C.Actual', ''):<34} {lag:>5}  {durum}")

print(f"\nkarar degiskeni : {len(dvs)}")
print(f"embargo (medyan): {splits.suggest_embargo(df, dvs)} satir")

## Sifirlar olcum degil, eksik veri

Output olcumlerindeki tam sifirlar sensor dropout'u; temizlikte NaN'a
cevrildi (R1). Gecerli verisi %50'nin altinda kalan output'lar kapsam
disi birakildi (R6), setpoint'i olcum gurultusunden ayirt edilemeyenler
de (R7). Kapsam disi kalanlar ve gerekceleri:

In [ ]:
disarida = kpi[kpi.in_scope != "evet"]
print(disarida[["output", "valid_pct", "scope_reason"]].to_string(index=False))
print(f"\nkapsam ici: {(kpi.in_scope == 'evet').sum()} / {len(kpi)}")

## Bias mi, variability mi?

Ikisi farkli muhendislik problemi: bias bir **ayar/kalibrasyon** sorunu,
variability bir **proses kontrol** sorunu. Toplam hatayi tek sayida
birlestirmek optimizasyonun yanlis seyi kovalamasina yol acar.

Asagidaki grafikte kosegenin ustunde kalanlar bias-baskin, altinda
kalanlar variability-baskin. Optimizasyonun hedefi ikinci gruptur;
bias-baskin bir output'u parametre oynatarak kovalamak yanlistir,
cozumu setpoint'i duzeltmektir.

In [ ]:
ins = kpi[kpi.in_scope == "evet"]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ins.dev_std, ins.bias.abs(), s=40)
for _, r in ins.iterrows():
    ax.annotate(r.output.replace("Stage", "S"), (r.dev_std, abs(r.bias)),
                fontsize=7, xytext=(3, 3), textcoords="offset points")

lim = max(ins.dev_std.max(), ins.bias.abs().max()) * 1.05
ax.plot([0, lim], [0, lim], lw=1, ls="--", color="gray")
ax.set_xlabel("dev_std — yayilim")
ax.set_ylabel("|bias| — merkezleme")
ax.set_title("Kosegenin ustu bias-baskin, alti variability-baskin")
plt.show()

## Buradan sonra

| Soru | Nerede |
|---|---|
| Veri kalitesi bulgulari | `reports/01_data_quality_report.md` |
| Temizlik kurallari (R1-R8) | `reports/02_cleaning_report.md` |
| Kararlilik ve Cp/Cpk | `reports/03_capability_report.md` |
| Korelasyon + n_eff/FDR duzeltmesi | `reports/04_correlation_report.md` |
| Varsayimlar ve kisitlar (A1, K1-K18) | `docs/assumptions.md` |

Burada bir sey dikkatini cekerse, once ilgili script'e tasi ve testini
yaz. Defterde gorulen bir orunti, bir script tarafindan yeniden
uretilmeden rapora girmez.